# GeoSentinel-AI Final Training (Phase 5)
This notebook trains BOTH the Baseline ImageNet U-Net AND the Final "Elite" Semantic U-Net sequentially on Kaggle.

### Pre-requisites:
1. You must upload your `deeplabv3plus_elite.pt` file to Kaggle as a **New Dataset**.
2. Make sure you clicked 'Add Input' on the right panel to attach it.

In [ ]:
!pip install torch torchvision torchgeo lightning segmentation-models-pytorch rasterio pystac-client planetary-computer

In [ ]:
import os
import shutil

# Force a fresh clone to guarantee we have the latest bug fixes
if os.path.exists('GeoSentinel-AI'):
    shutil.rmtree('GeoSentinel-AI')

!git clone https://github.com/karthikeya-bhamidipati/GeoSentinel-AI.git
os.chdir('GeoSentinel-AI')

In [ ]:
# Auto-detect the Kaggle dataset path containing your Elite weights
from pathlib import Path

def find_elite_weights(start_path="/kaggle/input"):
    for root, _, files in os.walk(start_path):
        if "deeplabv3plus_elite.pt" in files:
            return os.path.join(root, "deeplabv3plus_elite.pt")
    return None

elite_path = find_elite_weights()

if elite_path:
    print(f"Found Elite weights at: {elite_path}")
    os.makedirs('data/weights', exist_ok=True)
    # Rename it to 'best.pt' so the script automatically loads it
    shutil.copy(elite_path, 'data/weights/deeplabv3plus_best.pt')
    print("Elite weights successfully staged!")
else:
    print("ERROR: Could not find 'deeplabv3plus_elite.pt' in /kaggle/input!")
    print("Did you definitely upload it as a dataset and click 'Add Input'?")

In [ ]:
# 1. Train the Baseline ImageNet Control (Ablation Mode)
print("Starting Phase 1: Baseline ImageNet Model...")
!python scripts/train_change.py --epochs 150 --batch-size 8 --ablation

In [ ]:
# 2. Train the Final Elite Semantic Model
print("Starting Phase 2: Elite Semantic Model...")
!python scripts/train_change.py --epochs 150 --batch-size 8

In [ ]:
from IPython.display import FileLink

if os.path.exists('data/weights/change_unet_baseline_best.pt'):
    shutil.copy('data/weights/change_unet_baseline_best.pt', '/kaggle/working/change_unet_baseline_best.pt')
    print("\n--- DOWNLOAD BASELINE WEIGHTS ---")
    display(FileLink(r'/kaggle/working/change_unet_baseline_best.pt'))

if os.path.exists('data/weights/change_unet_best.pt'):
    shutil.copy('data/weights/change_unet_best.pt', '/kaggle/working/change_unet_elite_best.pt')
    print("\n--- DOWNLOAD ELITE SEMANTIC WEIGHTS ---")
    display(FileLink(r'/kaggle/working/change_unet_elite_best.pt'))